In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, pathlib
REPO_DIR = pathlib.Path(os.path.dirname(os.path.abspath(os.getcwd()))); print(f"Repo: {str(REPO_DIR)}")
sys.path.insert(0, str(REPO_DIR / "src"))

from tabulate import tabulate
from traffic_flow.types import *

Repo: /Users/tristanchaang/MIT/UROP-Traffic/mpc-traffic-flow


# A tiny Pyomo control problem

Fill a bucket from **0 to 3 litres in $N$ steps**. At each step, choose an inflow between 0 and 2 litres.

- State: $L_s$ — water already in the bucket. $(0\leq s \leq N)$
- Control: $0 \leq q_t \leq 2$ — water added during step $t$. $(0\leq t < N)$
- Dynamics: $L_{t+1} = L_t + q_t$. $(0 \leq t < N)$
- Constraints: $L_0 = 0, L_N = 3$.
- Objective: $\min \displaystyle\sum_{t=0}^{N-1} q_t^2$.


In [2]:
class BucketModel(PyoModel):
    s: FiniteScalarRangeSet; t: FiniteScalarRangeSet; 
    L: IndexedVar; q: IndexedVar
    dynamics: IndexedConstraint; constraints: ConstraintList; 
    objective: ScalarObjective

N = 6
MAX_INFLOW = 2.0

model = BucketModel()
model.s, model.t = (FiniteScalarRangeSet(0, k-1) for k in [N + 1, N])
model.L, model.q = (IndexedVar(r, within=pyo.NonNegativeReals, bounds=b, initialize=0)
                    for r, b in zip([model.s, model.t], [None, (0, MAX_INFLOW)]))
model.dynamics = IndexedConstraint(model.t, rule=lambda m, t: m.L[t + 1] == m.L[t] + m.q[t])
model.constraints = ConstraintList()
model.constraints.add(model.L[0] == 0)    # Start empty.
model.constraints.add(model.L[N] == 3.0)  # Finish at exactly 3 litres.
model.objective = ScalarObjective(sense = pyo.minimize, expr = sum((t+1) * model.q[t] ** 2 for t in model.t))

status = pyo.SolverFactory("ipopt").solve(model, tee=False, load_solutions=False, options={"max_iter": 100, "tol": 1e-9})

assert status.solver.status in (pyo.SolverStatus.ok, pyo.SolverStatus.warning)
assert status.solver.termination_condition == pyo.TerminationCondition.optimal
assert len(status.solution) > 0

model.solutions.load_from(status) # we didn't load solutions so that we can check success before loading 

rows = [(t, pyo.value(model.L[t]), pyo.value(model.q[t]), pyo.value(model.L[t + 1]))
        for t in model.t]
print(tabulate(rows, floatfmt=".2f", tablefmt="outline", headers=["Step", "Start level", "Inflow", "End level"]))
print(f"Total effort: {pyo.value(model.objective):.2f}")

# model.pprint()


+--------+---------------+----------+-------------+
|   Step |   Start level |   Inflow |   End level |
+========+===============+==========+=============+
|      0 |         -0.00 |     1.22 |        1.22 |
|      1 |          1.22 |     0.61 |        1.84 |
|      2 |          1.84 |     0.41 |        2.24 |
|      3 |          2.24 |     0.31 |        2.55 |
|      4 |          2.55 |     0.24 |        2.80 |
|      5 |          2.80 |     0.20 |        3.00 |
+--------+---------------+----------+-------------+
Total effort: 3.67


# A tiny Pyomo calibration problem

$m=2$ taps fill separate buckets. Their $N$ measured positive levels are known; estimate each tap's **constant flow rate**. Unlike the control example, the unknowns are system parameters, not time-varying controls.

- $0\leq t \leq N$, $0 \leq i < m$.
- Sampling interval $T = 0.5$ seconds.
- Measured levels $\hat L_{t,i}$ in litres. $(0\leq t \leq N$, $0 \leq i < m)$
- Fitted variables: $q_i$ — flow rate of tap $i$ in litres/second.
- Dynamics: $L_{t,i} = L_{t-1,i} + Tq_i$, with $L_{0,i}=\hat L_{0,i}$.
- Objective: $\min \displaystyle\sum_{t=0}^{N}\sum_i (L_{t,i}-\hat L_{t,i})^2$.

Expected fitted rates: **1 and 2 litres/second**, with zero squared error.


In [3]:
class TapCalibrationModel(PyoModel):
    num_taps: int
    t: FiniteScalarRangeSet; i: FiniteScalarRangeSet
    T: ScalarParam; L_hat: IndexedParam
    q: IndexedVar; L: IndexedVar
    dynamics: IndexedConstraint; constraints: ConstraintList
    loss: ScalarObjective

LEVELS = [(0.0, 0.0), (0.5, 1.0), (1.0, 2.0), (1.5, 3.0)]
N = len(LEVELS) - 1

model = TapCalibrationModel()
model.num_taps = len(LEVELS[0])
model.t, model.i = (FiniteScalarRangeSet(0, k) for k in [N, model.num_taps - 1])
model.T = ScalarParam(initialize=0.5)  # Known sampling interval; not fitted.
model.L_hat = IndexedParam(model.t, model.i, initialize={(t, i): LEVELS[t][i] for t in model.t for i in model.i})
model.q = IndexedVar(model.i, within=pyo.NonNegativeReals, bounds=(0, 3), initialize=1)
model.L = IndexedVar(model.t, model.i, within=pyo.NonNegativeReals, initialize=0)
model.constraints = ConstraintList()
for i in model.i: model.constraints.add(model.L[0, i] == model.L_hat[0, i])
model.dynamics = IndexedConstraint(model.t, model.i, 
    expr = {(t, i): pyo.Constraint.Skip if t == 0 else model.L[t, i] == model.L[t - 1, i] + model.T * model.q[i] 
           for t in model.t for i in model.i})
model.loss = ScalarObjective(sense=pyo.minimize,
    expr = sum((model.L[t, i] - model.L_hat[t, i]) ** 2 for t in model.t for i in model.i))

status = pyo.SolverFactory("ipopt").solve(model, tee=False, load_solutions=False, options={"max_iter": 100, "tol": 1e-9})

assert status.solver.status in (pyo.SolverStatus.ok, pyo.SolverStatus.warning)
assert status.solver.termination_condition == pyo.TerminationCondition.optimal
assert len(status.solution) > 0

model.solutions.load_from(status)  # Check success before loading, as in the control example.

rows = [(t, i, pyo.value(model.L_hat[t, i]), pyo.value(model.L[t, i]))
        for t in model.t for i in model.i]
print(tabulate(rows, headers=["Step", "Tap", "Measured level", "Fitted level"],
                floatfmt=".2f", tablefmt="outline"))
print(f"Fitted flow rates: {[pyo.value(model.q[i]) for i in model.i]} litres/second")
print(f"Total squared error: {pyo.value(model.loss):.2f}")

# model.pprint()


+--------+-------+------------------+----------------+
|   Step |   Tap |   Measured level |   Fitted level |
+========+=======+==================+================+
|      0 |     0 |             0.00 |          -0.00 |
|      0 |     1 |             0.00 |          -0.00 |
|      1 |     0 |             0.50 |           0.50 |
|      1 |     1 |             1.00 |           1.00 |
|      2 |     0 |             1.00 |           1.00 |
|      2 |     1 |             2.00 |           2.00 |
|      3 |     0 |             1.50 |           1.50 |
|      3 |     1 |             3.00 |           3.00 |
+--------+-------+------------------+----------------+
Fitted flow rates: [1.000000000045454, 2.0000000000129865] litres/second
Total squared error: 0.00


In [ ]:
from traffic_flow import load_results, save_result, evaluate, load_params
from traffic_flow.config import CalSource, CalRef, Study

r = load_results("i24", "11_30", calibration=CalRef(CalSource.FIXED_RAMPS, interval = None), study=Study.SAFETY_SWEEP)
r[0].scenario.spec.to_mpc_config()
# r[0].config.initialize_vsl

In [ ]:
import json as js
from traffic_flow.inputs.scenario import load_scenario

def print_dict(d):
  for key, value in d.items():
    print(f"{key}: {value}")

scenario = load_scenario("i24", "11_30")
params = load_params(scenario, CalRef(CalSource.FIXED_RAMPS, interval = None))

study = Study.SAFETY_SWEEP
directory = REPO_DIR / "results_bu" / "i24" / f"i24_11_30" / CalSource.FIXED_RAMPS / study

for npy in sorted(directory.glob("**/*.npy")):
  if "test" in npy.stem: continue
  json = npy.with_name(f"{npy.stem}_config.json")
  if "test" in str(json): continue
  with open(json, "r") as f:
    # print(json)
    data = js.load(f)
    del data["initialize_vsl"]

  vsl = np.load(npy, allow_pickle=False)

  mpc_config = scenario.spec.to_mpc_config(**data)
  rr = evaluate(scenario, params, vsl)

  save_result(RunResult(scenario, params, mpc_config, rr), CalRef(CalSource.FIXED_RAMPS, interval = None), study, f"temp_{mpc_config.safety_temporal}_spat_{mpc_config.safety_spatial}")

No dynamic parameter subfolders found — using static calibration parameters
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.1_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.2_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.3_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.4_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.5_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.7_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_0.9_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_1.5_spat_25.npz
Saved to results/i24/i24_11_30/calibration_static/fixed_ramping/safety_sweep/temp_10_spat_1.8.npz
Saved to results/i24/i24_11_30/calibration